<a href="https://colab.research.google.com/github/maggiecrowner/DS5001-Final-Project/blob/main/ParsedandAnnotatedData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Parsed and Annotated Data

In [1]:
! git clone https://github.com/maggiecrowner/DS5001-Final-Project.git

Cloning into 'DS5001-Final-Project'...
remote: Enumerating objects: 136, done.
remote: Counting objects: 100% (136/136), done.
remote: Compressing objects: 100% (128/128), done.
remote: Total 136 (delta 33), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (136/136), 11.19 MiB | 4.18 MiB/s, done.
Resolving deltas: 100% (33/33), done.


In [3]:
import pandas as pd
import io
import requests

box_files = {
    "PostMalone.csv":    "https://virginia.box.com/shared/static/o6v4vb3c3pe7kwysmbho7e8agi1co4z4.csv",
    "TaylorSwift.csv":   "https://virginia.box.com/shared/static/ynww2btxjuvl1cei9yaawxm0jgkrqvfs.csv",
    "EdSheeran.csv":     "https://virginia.box.com/shared/static/6zgbpge5yklaiiyypq62eif15axxrm07.csv",
    "JustinBieber.csv":  "https://virginia.box.com/shared/static/k92hby4d8pcbqc2ka9kpeicu4wv0jaiw.csv",
    "ColdPlay.csv":      "https://virginia.box.com/shared/static/2mb65mgblh008fg5s6sfio3bdey1eiph.csv",
    "Maroon5.csv":       "https://virginia.box.com/shared/static/v0az2kge2s8lycapyvzsmx813bb6avkt.csv",
    "Beyonce.csv":       "https://virginia.box.com/shared/static/3giiurqlx8jrs1t83gx0fisstxzv0wy8.csv",
    "LadyGaga.csv":      "https://virginia.box.com/shared/static/uzwtc77cxjauk2vn5xoeqms50prw5of9.csv",
    "BillieEilish.csv":  "https://virginia.box.com/shared/static/airnc5s6mw6c3zrieqps4e3nxc0stk0m.csv",
    "ArianaGrande.csv":  "https://virginia.box.com/shared/static/8m1ekqm8l303le447r3vzgpn773xo9pv.csv",
}

dfs = []
for filename, url in box_files.items():
    df = pd.read_csv(url)
    df['_source_file'] = filename
    dfs.append(df)
    print(f"Loaded {filename}")

source_data = pd.concat(dfs, ignore_index=True)
print(f"\nCorpus shape: {source_data.shape}")
source_data.head()

Loaded PostMalone.csv
Loaded TaylorSwift.csv
Loaded EdSheeran.csv
Loaded JustinBieber.csv
Loaded ColdPlay.csv
Loaded Maroon5.csv
Loaded Beyonce.csv
Loaded LadyGaga.csv
Loaded BillieEilish.csv
Loaded ArianaGrande.csv

Corpus shape: (3073, 8)


,Unnamed: 0,Artist,Title,Album,Year,Date,Lyric,_source_file
0,0.0,Post Malone,​​rockstar,beerbongs & bentleys,2017.0,2017-09-15,post malone hahahahaha tank god ayy ayy post...,PostMalone.csv
1,1.0,Post Malone,White Iverson,Stoney (Deluxe),2015.0,2015-02-04,double ot i'm a new three saucin' saucin' i'...,PostMalone.csv
2,2.0,Post Malone,Congratulations,Stoney (Deluxe),2016.0,2016-11-04,post malone mmmmm yeah yeah mmmmm yeah hey p...,PostMalone.csv
3,3.0,Post Malone,Psycho,beerbongs & bentleys,2018.0,2018-02-23,post malone damn my ap goin' psycho lil' mama ...,PostMalone.csv
4,4.0,Post Malone,I Fall Apart,Stoney (Deluxe),2016.0,2016-12-09,ooh i fall apart ooh yeah mmm yeah she told ...,PostMalone.csv


In [9]:
source_data = source_data[~source_data['Lyric'].str.contains('lyrics for this song have yet to be released please check back once the song has been released', case=False, na=False)]
print(source_data.shape)

(2960, 12)


## LIB

In [10]:
source_data['track_id'] = source_data['Artist'] + ' — ' + source_data['Title'] + ' (' + source_data['Album'] + ')'

# add additional columns for model summarization
source_data['doc_length_words'] = source_data['Lyric'].astype(str).str.split().str.len()
source_data['doc_length_chars'] = source_data['Lyric'].astype(str).str.len()
source_data['Year'] = pd.to_numeric(source_data['Year'], errors='coerce')
source_data['Decade'] = (source_data['Year'].dropna().astype(int) // 10 * 10).astype(str) + 's'

LIB = (source_data[['track_id', 'Artist', 'Album', 'Title', 'Year', 'Decade',
                     'doc_length_words', 'doc_length_chars']]
       .drop_duplicates()
       .set_index('track_id'))

print(f" LIB shape: {LIB.shape}")
LIB.head()

 LIB shape: (2960, 7)


,Artist,Album,Title,Year,Decade,doc_length_words,doc_length_chars
track_id,,,,,,,
Post Malone — ​​rockstar (beerbongs & bentleys),Post Malone,beerbongs & bentleys,​​rockstar,2017.0,2010s,507,2560
Post Malone — White Iverson (Stoney (Deluxe)),Post Malone,Stoney (Deluxe),White Iverson,2015.0,2010s,466,2324
Post Malone — Congratulations (Stoney (Deluxe)),Post Malone,Stoney (Deluxe),Congratulations,2016.0,2010s,533,2697
Post Malone — Psycho (beerbongs & bentleys),Post Malone,beerbongs & bentleys,Psycho,2018.0,2010s,536,2691
Post Malone — I Fall Apart (Stoney (Deluxe)),Post Malone,Stoney (Deluxe),I Fall Apart,2016.0,2010s,320,1552


In [12]:
LIB.to_csv('/content/DS5001-Final-Project/LIB.csv', sep='|', index=True)

## CORPUS

In [13]:
import nltk
import re
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('punkt')
nltk.download('punkt_tab')

pos_group_map = {
    'NN': 'NOUN', 'NNS': 'NOUN', 'NNP': 'NOUN', 'NNPS': 'NOUN',
    'VB': 'VERB', 'VBD': 'VERB', 'VBG': 'VERB', 'VBN': 'VERB', 'VBP': 'VERB', 'VBZ': 'VERB',
    'JJ': 'ADJ',  'JJR': 'ADJ',  'JJS': 'ADJ',
    'RB': 'ADV',  'RBR': 'ADV',  'RBS': 'ADV',
}

records = []
for _, row in source_data.iterrows():
    artist = row['Artist']
    album  = row['Album']
    track  = row['Title']
    lyric  = str(row['Lyric'])

    words    = lyric.split()
    pos_tags = nltk.pos_tag(words)

    for word_id, (token_str, pos) in enumerate(pos_tags):
        term_str  = re.sub(r'[^\w\s]', '', token_str).lower()
        pos_group = pos_group_map.get(pos, 'OTHER')

        records.append({
            'Artist':    artist,
            'Album':     album,
            'Title':     track,
            'WordID':    word_id,
            'token_str': token_str,
            'term_str':  term_str,
            'pos':       pos,
            'pos_group': pos_group,
        })

CORPUS = pd.DataFrame(records)
CORPUS = CORPUS.set_index(['Artist', 'Album', 'Title', 'WordID'])
print(f" Corpus shape: {CORPUS.shape}")
CORPUS.head()

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


 Corpus shape: (985352, 4)


token_str    term_str  \
Artist      Album                Title      WordID                           
Post Malone beerbongs & bentleys ​​rockstar 0             post        post   
                                            1           malone      malone   
                                            2       hahahahaha  hahahahaha   
                                            3             tank        tank   
                                            4              god         god   

                                                   pos pos_group  
Artist      Album                Title      WordID                
Post Malone beerbongs & bentleys ​​rockstar 0       NN      NOUN  
                                            1       NN      NOUN  
                                            2       NN      NOUN  
                                            3       NN      NOUN  
                                            4       NN      NOUN

In [14]:
CORPUS.to_csv('/content/DS5001-Final-Project/CORPUS.csv', sep='|', index=True)

## VOCAB

In [15]:
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import numpy as np

ps = PorterStemmer()

n = CORPUS['term_str'].value_counts().rename('n')
doc_term = CORPUS.reset_index().groupby('Title')['term_str'].apply(set)
N_docs = len(doc_term)

df_counts = (CORPUS.reset_index()
             .groupby('term_str')['Title']
             .nunique()
             .rename('df'))

VOCAB = pd.DataFrame({'n': n, 'df': df_counts})
VOCAB.index.name = 'term_str'

VOCAB['p']            = VOCAB['df'] / N_docs
VOCAB['i']            = np.log2(N_docs / VOCAB['df'])
VOCAB['dfidf']        = VOCAB['df'] * VOCAB['i']
VOCAB['porter_stem']  = VOCAB.index.map(lambda t: ps.stem(t))
VOCAB['max_pos']      = CORPUS.groupby('term_str')['pos'].agg(lambda x: x.value_counts().idxmax())
VOCAB['max_pos_group']= CORPUS.groupby('term_str')['pos_group'].agg(lambda x: x.value_counts().idxmax())
VOCAB['stop']         = VOCAB.index.map(lambda t: int(t in ENGLISH_STOP_WORDS))
VOCAB['ngram_length'] = VOCAB.index.map(lambda t: len(t.split()))
VOCAB = VOCAB.drop(columns=['df'])

print(f" VOCAB shape: {VOCAB.shape}")
VOCAB.head()

 VOCAB shape: (19104, 9)


,n,p,i,dfidf,porter_stem,max_pos,max_pos_group,stop,ngram_length
term_str,,,,,,,,,
,1,0.000345,11.501837,11.501837,,'',OTHER,0,0
0,151,0.030345,5.042406,443.731690,0,CD,OTHER,0,1
00,33,0.008621,6.857981,171.449525,00,CD,OTHER,0,1
000,14,0.002069,8.916875,53.501248,000,CD,OTHER,0,1
000s,1,0.000345,11.501837,11.501837,000,CD,OTHER,0,1


In [16]:
VOCAB.to_csv('/content/DS5001-Final-Project/VOCAB.csv', sep='|', index=True)

# EDA / Summary statistics of the tables

In [17]:
print("Average length of document, in characters:", source_data['Lyric'].str.len().mean())

Average length of document, in characters: 1642.7817500851208


In [18]:
print("Number of observations/tokens:", len(CORPUS['token_str']))

Number of observations/tokens: 985352


In [19]:
print("Number of observations/vocab:", len(VOCAB['dfidf']))

Number of observations/vocab: 19104


In [20]:
top20 = (VOCAB[VOCAB['stop'] == 0]
         .sort_values('dfidf', ascending=False)
         .head(20))
print("Top 20 significant words by DFIDF:")
top20[['n', 'p', 'i', 'dfidf', 'porter_stem', 'max_pos_group']]

Top 20 significant words by DFIDF:


,n,p,i,dfidf,porter_stem,max_pos_group
term_str,,,,,,
say,3023,0.368966,1.438442,1539.133051,say,VERB
baby,5378,0.380000,1.395929,1538.313401,babi,NOUN
time,2992,0.385172,1.376424,1537.465289,time,NOUN
want,4758,0.350000,1.514573,1537.291770,want,VERB
yeah,6556,0.387241,1.368695,1537.044454,yeah,NOUN
pre,2130,0.347586,1.524557,1536.753719,pre,NOUN
youre,4001,0.401724,1.315723,1532.817231,your,NOUN
got,4441,0.404828,1.304620,1531.624457,got,VERB
make,2513,0.324483,1.623786,1527.982882,make,VERB
